In [ ]:
# 初中物理试题生成（无检索版）—— 三种提示词策略对比
# 零样本 / 少样本 / 思维链

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
import os
import random

# ===================== 1. 环境配置 =====================
os.environ["DEEPSEEK_API_KEY"] = "sk-your-key-here"

llm = init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3
)

In [ ]:
# ===================== 2. 初中物理知识点字典 =====================
physics_map = {
    '机械运动': [
        '参照物', '运动相对性', '速度', '匀速直线运动',
        '变速运动', '平均速度', '路程-时间图像', '速度-时间图像'
    ],
    '声现象': [
        '声音的产生', '声音的传播', '声速', '回声',
        '声音的三个特性', '音调', '响度', '音色', '噪声', '噪声控制'
    ],
    '物态变化': [
        '温度', '温度计', '熔化和凝固', '凝固点', '汽化', '蒸发', '沸腾',
        '液化', '升华', '凝华', '晶体', '非晶体'
    ],
    '光现象': [
        '光的直线传播', '光速', '光的反射', '反射定律', '平面镜成像',
        '光的折射', '折射定律', '透镜', '凸透镜', '凹透镜',
        '凸透镜成像规律', '眼睛和眼镜', '显微镜和望远镜'
    ],
    '质量与密度': [
        '质量', '质量的测量', '天平', '密度', '密度公式',
        '密度计', '体积的测量', '物质鉴别'
    ],
    '力': [
        '力', '力的作用效果', '力的三要素', '弹力', '重力', '重心',
        '摩擦力', '滑动摩擦', '静摩擦', '滚动摩擦', '力的合成'
    ],
    '运动和力': [
        '牛顿第一定律', '惯性', '二力平衡', '摩擦力应用'
    ],
    '压强': [
        '压力', '压强', '液体压强', '大气压强', '连通器',
        '帕斯卡定律', '流速与压强', '浮力', '阿基米德原理', '物体的浮沉条件'
    ],
    '功和机械能': [
        '功', '功率', '动能', '势能', '机械能', '机械能守恒', '杠杆', '滑轮'
    ],
    '简单机械': [
        '杠杆平衡条件', '定滑轮', '动滑轮', '滑轮组', '机械效率'
    ],
    '电流和电路': [
        '电流', '电压', '电阻', '欧姆定律', '串联电路', '并联电路',
        '电流表', '电压表', '滑动变阻器', '电功', '电功率'
    ],
    '电与磁': [
        '磁体', '磁场', '磁感线', '电流的磁效应', '电磁铁', '电磁继电器',
        '电动机', '发电机', '电磁感应'
    ]
}

In [ ]:
# ===================== 3. 属性随机采样 =====================
difficulty_list = ['简单', '中等', '困难']
q_type_list = ['选择题', '填空题', '计算题']
calculation_list = ['不涉及运算', '涉及运算']
cognitive_levels_list = ['记忆', '理解', '应用', '分析', '评价', '创造']

def sample_attributes():
    """随机采样生成属性"""
    block = random.choice(list(physics_map.keys()))
    kp = random.choice(physics_map[block])
    difficulty = random.choice(difficulty_list)
    q_type = random.choice(q_type_list)
    calculation = random.choice(calculation_list)
    cognitive_level = random.choice(cognitive_levels_list)
    return block, kp, difficulty, q_type, calculation, cognitive_level

In [ ]:
# ===================== 4. 三种提示词模板 =====================

# -------- 4.1 零样本提示词 --------
prompt_zero_shot = PromptTemplate.from_template(
    '''
    你是一名初中物理资深命题教师，请根据以下任务说明进行出题。
    
    【已知条件】
    - 目标知识点：{knowledge_point}
    - 小节：{block}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：{cognitive_level}
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心
    2. 符合初中物理题风格
    3. 严格匹配给定的题型、难度、是否运算、认知层级
    4. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    5. 设问方式灵活，可以是填空、选择、计算、作图等
    6. 只生成一道题，不要解释、不输出思路
    
    严格按照以下格式（包括换行也要一致）输出，不要含有其他内容如前后缀等，最前面不要有空格
    【输出格式】：
    【{difficulty}题 | {q_type} | {cognitive_level}题 | {calculation}】
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）答案内容（计算题需写步骤）
    '''
)

# -------- 4.2 少样本提示词 --------
prompt_one_shot = PromptTemplate.from_template(
    '''
    你是一名初中物理资深命题教师。请严格根据以下要求出题。
    
    【示例1】
    已知条件：
    - 小节：物态变化
    - 目标知识点：汽化
    - 难度等级：简单
    - 题型：选择题
    - 是否涉及运算：否
    - 认知层级：记忆
    输出：
    【简单题 | 选择题 | 记忆题 | 不涉及运算】
    【题目】
    物质由液态变为气态的过程称为？
    【选项】
    A. 蒸发  B. 熔化  C. 凝固  D. 升华
    【参考答案】
    A
    
    【新任务】
    已知条件：
    - 小节：{block}
    - 目标知识点：{knowledge_point}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：{cognitive_level}
    
    请按照与示例相同的格式输出一道试题，只输出结果，不输出解释。
    【输出格式】：
    【{difficulty}题 | {q_type} | {cognitive_level}题 | {calculation}】
    【题目】
    （另起一行）题目内容
    （如是选择题）【选项】
    【参考答案】
    （另起一行）答案内容（计算题需写步骤）
    '''
)

# -------- 4.3 思维链提示词 --------
prompt_chain_of_thought = PromptTemplate.from_template(
    '''
    你是一名初中物理资深命题教师。请严格根据以下要求出题。
    
    【已知条件】
    - 目标知识点：{knowledge_point}
    - 小节：{block}
    - 难度等级：{difficulty}
    - 题型：{q_type}
    - 是否涉及运算：{calculation}
    - 认知层级（如：记忆 / 理解 / 应用 / 分析 / 评价 / 创造）：{cognitive_level}
    
    【出题步骤】
    请先按以下步骤思考，并将思考内容输出在【推理过程】中：
    步骤一：分析目标知识点的核心考查角度。
    步骤二：根据难度等级确定设问复杂程度与干扰项强度。
    步骤三：根据认知层级选择合适的设问动词：
           记忆层用"写出/列举"；理解层用"解释/概括"；应用层用"运用/解决"；
           分析层用"分析/比较"；评价层用"评价/论证"；创造层用"设计/提出"。
    步骤四：若涉及运算，设计数据条件与计算步骤；若不涉及，跳过此步。
    
    【出题要求】
    1. 题目必须以「目标知识点」为核心
    2. 严格匹配给定的题型、难度、是否运算、认知层级
    3. 题干表述清晰，不引入无关信息。纯文本题，不要用到图片。
    4. 设问方式灵活，可以是填空、选择、计算、作图等
    5. 符合初中物理试题风格
    6. 只生成一道题，不要输出额外解释
    
    【输出格式】
    【推理过程】
    （另起一行）依次输出上述四个步骤的思考内容，每个步骤用"步骤一："等前缀标明
    
    【{difficulty}题 | {q_type} | {cognitive_level}题 | {calculation}】
    【题目】
    （题目内容）
    【选项】（仅选择题需要）
    【参考答案】
    （答案内容，计算题需写步骤）
    '''
)

In [ ]:
# ===================== 5. 生成函数 =====================
def generate_question(strategy='zero_shot'):
    """
    单次生成：随机采样 + 调用 LLM
    
    参数：
        strategy: 'zero_shot' | 'one_shot' | 'chain_of_thought'
    """
    block, kp, difficulty, q_type, calculation, cognitive_level = sample_attributes()
    
    if strategy == 'zero_shot':
        prompt = prompt_zero_shot
    elif strategy == 'one_shot':
        prompt = prompt_one_shot
    elif strategy == 'chain_of_thought':
        prompt = prompt_chain_of_thought
    else:
        raise ValueError("strategy 必须是 'zero_shot' / 'one_shot' / 'chain_of_thought'")
    
    response = llm.invoke(
        prompt.format(
            knowledge_point=kp,
            block=block,
            difficulty=difficulty,
            q_type=q_type,
            calculation=calculation,
            cognitive_level=cognitive_level
        )
    )
    
    result = response.content.strip()
    print(f"策略：{strategy}")
    print(f"小节：{block} | 知识点：{kp}")
    print(f"属性：{difficulty} | {q_type} | {cognitive_level} | {calculation}")
    print(result)
    print("=" * 60)
    return result

In [ ]:
# ===================== 6. 批量生成 =====================
def generate_batch(n=10, strategy='zero_shot', save_path=None):
    """
    批量生成
    
    参数：
        n: 生成数量
        strategy: 'zero_shot' | 'one_shot' | 'chain_of_thought'
        save_path: 保存路径
    """
    if save_path is None:
        save_path = f'./physics_{strategy}.txt'
    
    with open(save_path, 'w', encoding='utf-8') as f:
        for i in range(n):
            print(f"正在生成第 {i+1} 题（策略：{strategy}）...")
            try:
                result = generate_question(strategy=strategy)
                f.write(result + "\n\n")
                f.flush()
            except Exception as e:
                print(f"生成失败: {e}")
    print(f"批量生成完成，共 {n} 题，保存至 {save_path}")

In [ ]:
# ===================== 使用示例 =====================
# generate_question(strategy='zero_shot')      # 零样本
# generate_question(strategy='one_shot')       # 少样本
# generate_question(strategy='chain_of_thought')  # 思维链
# generate_batch(n=30, strategy='one_shot', save_path='./physics_one_shot.txt')

In [ ]:
# ===================== 7. 评估数据读取 =====================
import pandas as pd
import numpy as np
from collections import Counter

def load_eval_data(file_path):
    """
    读取评估结果文件，提取三个指标：
    - 知识相关性：倒数第7个字段
    - 5次作答（用于计算可解释性）：倒数第6到倒数第2个字段
    - 难度准确性：倒数第1个字段
    
    文件格式示例：
    ...;0.8221;A;A;A;A;A;0.7628
    
    参数：
        file_path: txt文件路径，;分割，无表头
    返回：
        DataFrame，最后三列为 relevance, explainability, accuracy
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    
    results = []
    for line in lines:
        parts = line.split(';')
        
        # 提取三个指标
        relevance = float(parts[-7])          # 知识相关性
        answers = parts[-6:-1]               # 5次作答字母
        accuracy = float(parts[-1])         # 难度准确性
        
        # 计算可解释性：最常见答案出现次数 / 5
        valid_answers = [a for a in answers if a in 'ABCD']
        if valid_answers:
            counter = Counter(valid_answers)
            most_common_count = counter.most_common(1)[0][1]
            explainability = most_common_count / len(valid_answers)
        else:
            explainability = 0.0
        
        # 保留其他字段，加上三个指标
        other_fields = parts[:-7]
        new_row = other_fields + [relevance, explainability, accuracy]
        results.append(new_row)
    
    # 构建列名
    n_other = len(results[0]) - 3
    col_names = [f'field_{i}' for i in range(n_other)] + ['relevance', 'explainability', 'accuracy']
    
    df = pd.DataFrame(results, columns=col_names)
    return df

# -------- 使用示例 --------
# df = load_eval_data(r'C:\Users\lenovo\LLM\thesis\geograph\output\comparison\组合数据\你的文件名.txt')
# print(df[['relevance', 'explainability', 'accuracy']].describe())